In [ ]:
#EXTRACTING only RMS 

In [2]:
import torch
import librosa
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from pathlib import Path

In [3]:
filepaths = list(Path("daria-recordings").glob("*.wav"))
print(len(filepaths))
print(filepaths[:3])

18
[PosixPath('daria-recordings/419118__mrthenoronha__guitar-riff-heavy.wav'), PosixPath('daria-recordings/16188__pitx__wah_two_electric_guitar.wav'), PosixPath('daria-recordings/chitara1.wav')]


In [4]:
def extract_rms_windows(filepath, window_size=20, step=1):
    y, sr = librosa.load(str(filepath), sr=44100, mono=True)
    rms = librosa.feature.rms(y=y)[0]

    windows = []
    for i in range(0, len(rms) - window_size, step):
        window = rms[i:i + window_size]
        windows.append(window)

    return np.array(windows)

In [5]:
test_result = extract_rms_windows(filepaths[0])
print(test_result.shape)

(1672, 20)


In [6]:
all_tables = []

for fp in filepaths:
    table = extract_rms_windows(fp)
    all_tables.append(table)

feature_table = np.vstack(all_tables)
print(feature_table.shape)

/var/folders/8t/cmgzfn3515b7_cvsv4pbv7g00000gn/T/ipykernel_58457/3790979053.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(str(filepath), sr=44100, mono=True)
/Users/daria/opt/anaconda3/envs/c4dm/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


(55947, 20)


In [7]:
from sklearn import preprocessing

scaler = preprocessing.MinMaxScaler()
scaled_features = scaler.fit_transform(feature_table)

print(scaled_features.min(axis=0))
print(scaled_features.max(axis=0))

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [9]:
np.save("RMS-features.npy", scaled_features)